# $$\text{Modelado del Churn}$$

# REGRESIÓN LOGÍSTICA

## 0. Configuración inicial

In [1]:
# importar librerías

# Manipulación de datos
import pandas as pd
import numpy as np

# Manipulación de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Paths y automatizaciones
import joblib
import sys
from pathlib import  Path

# Módulos sklearn

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    make_scorer, recall_score, # para scoring personalizado en GridSearchCV
    precision_score, f1_score 
)

# Explicabilidad modelo
import shap

c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths

INPUT_PATH   = Path("data\processed\df_final.csv")
MODEL_PATH   = Path("models\pipeline.pkl")
# METRICS_PATH = Path("4_outputs/synthetic_data/2_metrics")
FIG_PATH     = Path("figures")

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
# METRICS_PATH.mkdir(parents=True, exist_ok=True)
FIG_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42

In [3]:
# Importar dataframe para modelado
df_final = pd.read_csv("../data/processed/df_final.csv")

In [4]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 43 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender_M                        200 non-null    float64
 1   Location_Canada                 200 non-null    float64
 2   Location_UK                     200 non-null    float64
 3   Location_USA                    200 non-null    float64
 4   CancelReason_Lack of use        200 non-null    float64
 5   CancelReason_No cancellation    200 non-null    float64
 6   CancelReason_Switched provider  200 non-null    float64
 7   PlanType_Premium                200 non-null    float64
 8   PlanType_Standard               200 non-null    float64
 9   UserID                          200 non-null    int64  
 10  Age                             200 non-null    int64  
 11  RegistrationDate_x              200 non-null    object 
 12  RegistrationDate_y              200 

## 1. Carga de dataframe de usuarios

In [6]:
# ── 1. Carga ──────────────────────────────────────────────────────────────────

df = df_final.copy()
print("=" * 60)
print("REPORTE DE MODELADO — CHURN PREDICTION")
print("=" * 60)
print(f"\nFilas cargadas  : {len(df):,}")
print(f"Tasa de churn   : {df['churned'].mean():.2%}")

REPORTE DE MODELADO — CHURN PREDICTION

Filas cargadas  : 200
Tasa de churn   : 27.00%


In [ ]:
# ── 2. Winsorización de variables con datos atípicos (percentil 99) ────────────────────────

# p99 = df["valor_total_6m"].quantile(0.99)
# df["valor_total_6m"] = df["valor_total_6m"].clip(upper=p99)
# print(f"\n── Winsorización valor_total_6m ───────────────────────────")
# print(f"  Percentil 99   : {p99:,.0f}")
# print(f"  Valor máximo post-winsorización: {df['valor_total_6m'].max():,.0f}")

## 2. Features del modelo

### 2.1. modelo 1

In [50]:
# ── 1. Definir X e y ──────────────────────────────────────────────────────────
EXCLUIR = ["UserID", "churned", "RegistrationDate_x", "RegistrationDate_y", "EndDate", "first_payment_date", "last_payment_date","first_contentviewdate", "last_contentviewdate", "first_helprqsdate", "last_helprqsdate",
           # Razones de cancelación
           "CancelReason_Lack of use", "CancelReason_No cancellation", "CancelReason_Switched provider"]
X = df.drop(columns=EXCLUIR)
y = df["churned"]

In [51]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Gender_M                       200 non-null    float64
 1   Location_Canada                200 non-null    float64
 2   Location_UK                    200 non-null    float64
 3   Location_USA                   200 non-null    float64
 4   PlanType_Premium               200 non-null    float64
 5   PlanType_Standard              200 non-null    float64
 6   Age                            200 non-null    int64  
 7   DurationMonths                 200 non-null    int64  
 8   total_payments                 200 non-null    int64  
 9   first_payment_amount           200 non-null    float64
 10  last_payment_amount            200 non-null    float64
 11  CampaignID                     200 non-null    int64  
 12  CampaignName                   200 non-null    obj

> 👇👇 ***AJUSTAR***

In [ ]:
# # -- 2. Columnas por tipo -------------------------------------------------------------
# NUMERICAS = [
#     "antiguedad_dias", "num_envios_6m", "valor_total_6m",
#     "ticket_promedio", "dias_ultimo_envio", "num_reclamos_6m",
#     "tasa_entrega_exitosa", "nps_score", "ratio_reclamos_envios"
# ]
# CATEGORICAS = [
#     "ciudad", "tipo_cliente", "canal_principal",
#     "tiene_contrato", "segmento_recencia"
# ]

# print(f"\n── Features del modelo ────────────────────────────────────")
# print(f"  Numéricas   ({len(NUMERICAS)}): {NUMERICAS}")
# print(f"  Categóricas ({len(CATEGORICAS)}): {CATEGORICAS}")

In [52]:
# 1. Separar por tipo de dato numérico
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 2. Separar las columnas que son fechas (object que contienen 'date')
# Esto es útil para separar las variables de "tiempo" de las "categóricas" reales
date_cols = [col for col in X.columns if 'date' in col.lower()]

# 3. Separar las categóricas (objetos que NO son fechas)
categorical_cols = [col for col in X.select_dtypes(include=['object']).columns 
                    if col not in date_cols]

# 4. Ajuste manual: Si quieres mover las dummies (que ahora son float) 
# a una lista de "dummies" separada de tus numéricas continuas:
dummy_cols = [col for col in numeric_cols if any(x in col for x in ['Gender', 'Location', 'CancelReason', 'PlanType'])]
numeric_continuous = [col for col in numeric_cols if col not in dummy_cols]

print(f"Numéricas continuas: {len(numeric_continuous)}")
print(f"Dummies: {len(dummy_cols)}")
print(f"Categóricas: {len(categorical_cols)}")
print(f"Fechas: {len(date_cols)}")

Numéricas continuas: 22
Dummies: 6
Categóricas: 1
Fechas: 0


In [53]:
# ── 3. Split train/test estratificado (70% train set / 30% test set) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30, # 30% para test, 70% para train
    random_state=SEED,
    stratify=y # Mantener proporción de churn en ambos sets
)
print(f"\n── Split train/test ───────────────────────────────────────")
print(f"  Train : {len(X_train):,} filas | churn={y_train.mean():.2%}")
print(f"  Test  : {len(X_test):,}  filas | churn={y_test.mean():.2%}")


── Split train/test ───────────────────────────────────────
  Train : 140 filas | churn=27.14%
  Test  : 60  filas | churn=26.67%


> **Column transformer**

- Se aplicó previamente el `OneHotEncoder` a las columnas categóricas.
- Sólo se aplicará a través del *Column Transformer*, el `StandardScaler` a las columnas con valores numéricos contínuos

In [54]:
# Limpiar listas de variables numéricas continuas (remover IDs y target)
# numericas_remover = ['churned']

# for col in numericas_remover:
#     numeric_continuous.remove(col)

numeric_continuous

['Age',
 'DurationMonths',
 'total_payments',
 'first_payment_amount',
 'last_payment_amount',
 'CampaignID',
 'CampaignCost',
 'UsersAcquired',
 'CAC',
 'total_interactions',
 'number_helprqs',
 'number_contentviews',
 'number_views_genreAction',
 'number_views_genreComedy',
 'number_views_genreDocumentary',
 'number_views_genreDrama',
 'number_views_genreHorror',
 'number_views_genreRomance',
 'number_views_genreThriller',
 'lifetimevalue',
 'ltvcac_ratio',
 'netcustomervalue']

In [55]:
# ── 4. ColumnTransformer ──────────────────────────────────────────────────────
preprocesamiento = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_continuous),
    ("dummies", "passthrough", dummy_cols) # Mantiene las dummies sin escalar
])

### 2.2. Modelo 2

In [66]:
df.columns

Index(['Gender_M', 'Location_Canada', 'Location_UK', 'Location_USA',
       'CancelReason_Lack of use', 'CancelReason_No cancellation',
       'CancelReason_Switched provider', 'PlanType_Premium',
       'PlanType_Standard', 'UserID', 'Age', 'RegistrationDate_x',
       'RegistrationDate_y', 'EndDate', 'DurationMonths', 'total_payments',
       'first_payment_date', 'last_payment_date', 'first_payment_amount',
       'last_payment_amount', 'CampaignID', 'CampaignName', 'CampaignCost',
       'UsersAcquired', 'CAC', 'total_interactions', 'number_helprqs',
       'number_contentviews', 'first_contentviewdate', 'last_contentviewdate',
       'first_helprqsdate', 'last_helprqsdate', 'number_views_genreAction',
       'number_views_genreComedy', 'number_views_genreDocumentary',
       'number_views_genreDrama', 'number_views_genreHorror',
       'number_views_genreRomance', 'number_views_genreThriller',
       'lifetimevalue', 'ltvcac_ratio', 'netcustomervalue', 'churned'],
      dtype='obj

In [67]:
# ── 1. Definir X e y ──────────────────────────────────────────────────────────
X_2 = df_final[[
         "PlanType_Premium",
         "PlanType_Standard",
         "Age",
         "number_contentviews",
         "number_helprqs",
         "lifetimevalue",
         "ltvcac_ratio"
         ]]
y = df_final["churned"]

In [68]:
X_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PlanType_Premium     200 non-null    float64
 1   PlanType_Standard    200 non-null    float64
 2   Age                  200 non-null    int64  
 3   number_contentviews  200 non-null    int64  
 4   number_helprqs       200 non-null    int64  
 5   lifetimevalue        200 non-null    float64
 6   ltvcac_ratio         200 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 11.1 KB


In [69]:
# 1. Separar por tipo de dato numérico
numeric_cols_2 = X_2.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 2. Separar las columnas que son fechas (object que contienen 'date')
# Esto es útil para separar las variables de "tiempo" de las "categóricas" reales
date_cols_2 = [col for col in X_2.columns if 'date' in col.lower()]

# 3. Separar las categóricas (objetos que NO son fechas)
categorical_cols_2 = [col for col in X_2.select_dtypes(include=['object']).columns 
                    if col not in date_cols_2]

# 4. Ajuste manual: Si quieres mover las dummies (que ahora son float) 
# a una lista de "dummies" separada de tus numéricas continuas:
dummy_cols_2 = [col for col in numeric_cols_2 if any(x in col for x in ['Gender', 'Location', 'CancelReason', 'PlanType'])]
numeric_continuous_2 = [col for col in numeric_cols_2 if col not in dummy_cols_2]

print(f"Numéricas continuas: {len(numeric_continuous_2)}")
print(f"Dummies: {len(dummy_cols_2)}")
print(f"Categóricas: {len(categorical_cols_2)}")
print(f"Fechas: {len(date_cols_2)}")


Numéricas continuas: 5
Dummies: 2
Categóricas: 0
Fechas: 0


In [70]:
# ── 3. Split train/test estratificado (70% train set / 30% test set) ─────────────────────────────────
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_2, y,
    test_size=0.30, # 30% para test, 70% para train
    random_state=SEED,
    stratify=y # Mantener proporción de churn en ambos sets
)
print(f"\n── Split train/test ───────────────────────────────────────")
print(f"  Train : {len(X_train_2):,} filas | churn={y_train_2.mean():.2%}")
print(f"  Test  : {len(X_test_2):,}  filas | churn={y_test_2.mean():.2%}")


── Split train/test ───────────────────────────────────────
  Train : 140 filas | churn=27.14%
  Test  : 60  filas | churn=26.67%


In [71]:
# ── 4. ColumnTransformer ──────────────────────────────────────────────────────
preprocesamiento_2 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_continuous_2),
    ("dummies", "passthrough", dummy_cols_2) # Mantiene las dummies sin escalar
])

## 3. Entrenamiento del modelo

- Se usa GridSearchCV para encontrar los mejores hiperparámetros de la regresión logística
- priorizamos `recall score` para capturar la mayor cantidad de churners posibles
- Se utiliza `cross-validation`, para robustecer el proceso de aprendizaje

### 3.1. modelo 1

In [56]:
# ── 1. Pipeline + GridSearchCV ────────────────────────────────────────────────
# Se usa GridSearchCV para encontrar los mejores hiperparámetros de la regresión 
# logística
pipeline = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("modelo", GridSearchCV(
        estimator=LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=1000,
        random_state=SEED
    ),
        param_grid={
            "C"        : [0.01, 0.1, 1, 10, 100],
            "l1_ratio" : [0.1, 0.5, 0.9] # 0.0=L2, 1.0=L1, 0.5=ElasticNet
        },
        scoring=make_scorer(f1_score, pos_label=1),
 # priorizamos recall para capturar la mayor cantidad de churners posibles
        cv=5, # 5-fold cross-validation, robustece el aprendizaje
        n_jobs=-1,
        verbose=1,
    ))
])

print(f"\n── Entrenando pipeline (GridSearchCV, 5-fold cross-validation, scoring=recf1_score)...")
pipeline.fit(X_train, y_train)


── Entrenando pipeline (GridSearchCV, 5-fold cross-validation, scoring=recf1_score)...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('dummies', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [57]:
# ── 2. Extraer los Mejores parámetros ─────────────────────────────────────────────────────
best_params = pipeline.named_steps["modelo"].best_params_
best_score  = pipeline.named_steps["modelo"].best_score_
print(f"\n── Mejores parámetros ─────────────────────────────────────")
for k, v in best_params.items():
    print(f"  {k:12s}: {v}")
print(f"  Recall CV (train): {best_score:.4f}")


── Mejores parámetros ─────────────────────────────────────
  C           : 1
  l1_ratio    : 0.1
  Recall CV (train): 0.1116


### 3.2. modelo 2

In [72]:
# ── 1. Pipeline + GridSearchCV ────────────────────────────────────────────────
# Se usa GridSearchCV para encontrar los mejores hiperparámetros de la regresión 
# logística
pipeline_2 = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento_2),
    ("modelo", GridSearchCV(
        estimator=LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=1000,
        random_state=SEED
    ),
        param_grid={
            "C"        : [0.01, 0.1, 1, 10, 100],
            "l1_ratio" : [0.1, 0.5, 0.9] # 0.0=L2, 1.0=L1, 0.5=ElasticNet
        },
        scoring=make_scorer(f1_score, pos_label=1),
 # priorizamos recall para capturar la mayor cantidad de churners posibles
        cv=5, # 5-fold cross-validation, robustece el aprendizaje
        n_jobs=-1,
        verbose=1,
    ))
])

print(f"\n── Entrenando pipeline (GridSearchCV, 5-fold cross-validation, scoring=recall)...")
pipeline_2.fit(X_train_2, y_train_2)


── Entrenando pipeline (GridSearchCV, 5-fold cross-validation, scoring=recall)...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('dummies', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [73]:
# ── 2. Extraer los Mejores parámetros ─────────────────────────────────────────────────────
best_params_2 = pipeline_2.named_steps["modelo"].best_params_
best_score_2  = pipeline_2.named_steps["modelo"].best_score_
print(f"\n── Mejores parámetros ─────────────────────────────────────")
for k, v in best_params_2.items():
    print(f"  {k:12s}: {v}")
print(f"  Recall CV (train): {best_score_2:.4f}")


── Mejores parámetros ─────────────────────────────────────
  C           : 10
  l1_ratio    : 0.1
  Recall CV (train): 0.1371


## 4. Evaluación del modelo

### 4.1. modelo 1

In [58]:
# ── 1. Evaluación del modelo en test set ─────────────────────────────────────────────────
y_pred      = pipeline.predict(X_test)
y_prob      = pipeline.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_prob)

print(f"\n── Métricas en test ───────────────────────────────────────")
print(f"  AUC-ROC : {auc:.4f}")
print(f"\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Activo","Churned"]))


── Métricas en test ───────────────────────────────────────
  AUC-ROC : 0.4702

Matriz de confusión:
[[39  5]
 [13  3]]

Classification Report:
              precision    recall  f1-score   support

      Activo       0.75      0.89      0.81        44
     Churned       0.38      0.19      0.25        16

    accuracy                           0.70        60
   macro avg       0.56      0.54      0.53        60
weighted avg       0.65      0.70      0.66        60



In [59]:
# ── 2. Matriz de confusión ─────────────────────────────────────────────────

cm       = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

In [62]:
# Ver correlación de todas tus features vs el target
correlaciones = X.drop(columns = 'CampaignName').corrwith(y).sort_values(ascending=False)
print(correlaciones)

Gender_M                         6.757374e-02
PlanType_Premium                 6.511660e-02
Location_UK                      1.069147e-02
Location_USA                     8.234244e-03
Location_Canada                  1.728698e-03
CAC                             -5.501607e-17
PlanType_Standard               -4.001120e-03
number_views_genreDrama         -7.913245e-03
total_payments                  -1.326758e-02
number_views_genreThriller      -2.415445e-02
Age                             -2.457737e-02
number_views_genreRomance       -2.724158e-02
number_views_genreHorror        -3.583794e-02
number_views_genreDocumentary   -3.763115e-02
netcustomervalue                -6.724653e-02
lifetimevalue                   -6.724653e-02
ltvcac_ratio                    -6.725650e-02
DurationMonths                  -8.347460e-02
first_payment_amount            -9.249744e-02
last_payment_amount             -9.249744e-02
number_contentviews             -9.812055e-02
number_views_genreAction        -1

c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\numpy\lib\_function_base_impl.py:3063: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\numpy\lib\_function_base_impl.py:3064: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


### 4.2. Modelo 2

In [74]:
# ── 1. Evaluación del modelo en test set ─────────────────────────────────────────────────
y_pred_2      = pipeline_2.predict(X_test_2)
y_prob_2      = pipeline_2.predict_proba(X_test_2)[:, 1]
auc_2         = roc_auc_score(y_test_2, y_prob_2)

print(f"\n── Métricas en test ───────────────────────────────────────")
print(f"  AUC-ROC : {auc_2:.4f}")
print(f"\nMatriz de confusión:")
print(confusion_matrix(y_test_2, y_pred_2))
print(f"\nClassification Report:")
print(classification_report(y_test_2, y_pred_2, target_names=["Activo","Churned"]))

# ── 2. Matriz de confusión ─────────────────────────────────────────────────

cm       = confusion_matrix(y_test_2, y_pred_2)
tn_2, fp_2, fn_2, tp_2 = cm.ravel()


── Métricas en test ───────────────────────────────────────
  AUC-ROC : 0.4773

Matriz de confusión:
[[38  6]
 [13  3]]

Classification Report:
              precision    recall  f1-score   support

      Activo       0.75      0.86      0.80        44
     Churned       0.33      0.19      0.24        16

    accuracy                           0.68        60
   macro avg       0.54      0.53      0.52        60
weighted avg       0.64      0.68      0.65        60



## 5. Métricas de negocio

> 👇👇 **Ajustar**

> **Métricas de impacto para el negocio - Supuestos**

In [48]:
# ── 2. Métricas de impacto para el negocio ───────────────────────────────────
print(f"\n── Métricas de impacto de negocio ─────────────────────────")
# NOTA: Los siguientes valores son supuestos estimados.
# En producción deben ser provistos por el área financiera/comercial
# de Interrapidísimo.
SUPUESTOS = {
# se puede calcular como el valor total de compras en 6 meses multiplicado por un factor de retención anual
# (ej. 2x para estimar 1 año)
"LTV_PROMEDIO"      : df["valor_total_6m"].median() * 2,  # estimado anual
# COP por cliente contactado por multiples canales (teléfono, email, SMS)
# Tiempo agente call center (15 min) = 7,000 COP
# Descuento o incentivo ofrecido = 5,000 COP
# Costo operativo (SMS, email, sistema) = 3,000 COP
"COSTO_INTERVENCION" : 15_000,
"TASA_RETENCION"      : 0.30,     # % de clientes contactados que se retienen
}

print(f"\n── Supuestos de negocio (configurables) ───────────────────")
for k, v in SUPUESTOS.items():
    print(f"  {k:25s}: {v:,.2f}")


── Métricas de impacto de negocio ─────────────────────────


KeyError: 'valor_total_6m'

In [ ]:
# Métricas de negocio
clientes_contactados = tp + fp
clientes_retenidos   = int(clientes_contactados * SUPUESTOS["TASA_RETENCION"])
costo_total          = clientes_contactados * SUPUESTOS["COSTO_INTERVENCION"]
ingreso_retenido     = clientes_retenidos   * SUPUESTOS["LTV_PROMEDIO"]
roi                  = (ingreso_retenido - costo_total) / costo_total * 100

print(f"\n── Métricas de impacto ────────────────────────────────────")
print(f"  Clientes churn detectados (TP) : {tp:,}")
print(f"  Falsos positivos (FP)          : {fp:,}")
print(f"  Clientes a contactar           : {clientes_contactados:,}")
print(f"  Clientes retenidos estimados   : {clientes_retenidos:,}")
print(f"  Costo total campaña            : ${costo_total:,.0f} COP")
print(f"  Ingreso retenido estimado      : ${ingreso_retenido:,.0f} COP")
print(f"  ROI estimado del modelo        : {roi:.1f}%")
print(f"\n  ⚠ Supuestos estimados — validar con áreas comercial y financiera para mayor precisión en métricas de negocio")


## 6. Gráficas

### 6.1. Modelo 1

In [75]:
# ── 11. Gráficas ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Evaluación del Modelo — Churn Prediction", fontsize=13, fontweight="bold")

# 11.1 Matriz de confusión
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Activo","Churn"],
    yticklabels=["Activo","Churn"],
    ax=axes[0]
)
axes[0].set_title("Matriz de Confusión")
axes[0].set_ylabel("Real")
axes[0].set_xlabel("Predicho")

# 11.2 Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_title("Curva ROC")
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].legend()

# 11.3 Curva Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(recall, precision, color="#3498db", lw=2)
axes[2].set_title("Curva Precision-Recall")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")

plt.tight_layout()
fig.savefig(FIG_PATH / "modelo1_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nGráfica guardada en: {FIG_PATH / 'modelo1_model_evaluation.png'}")


Gráfica guardada en: figures\modelo1_model_evaluation.png


### 6.2. Modelo 2

In [76]:
# ── 11. Gráficas ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Evaluación del Modelo — Churn Prediction", fontsize=13, fontweight="bold")

# 11.1 Matriz de confusión
sns.heatmap(
    confusion_matrix(y_test_2, y_pred_2),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Activo","Churn"],
    yticklabels=["Activo","Churn"],
    ax=axes[0]
)
axes[0].set_title("Matriz de Confusión")
axes[0].set_ylabel("Real")
axes[0].set_xlabel("Predicho")

# 11.2 Curva ROC
fpr, tpr, _ = roc_curve(y_test_2, y_prob_2)
axes[1].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_title("Curva ROC")
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].legend()

# 11.3 Curva Precision-Recall
precision, recall, _ = precision_recall_curve(y_test_2, y_prob_2)
axes[2].plot(recall, precision, color="#3498db", lw=2)
axes[2].set_title("Curva Precision-Recall")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")

plt.tight_layout()
fig.savefig(FIG_PATH / "modelo2_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nGráfica guardada en: {FIG_PATH / 'modelo2_model_evaluation.png'}")


Gráfica guardada en: figures\modelo2_model_evaluation.png


## 7. Pipeline

In [77]:
# ── 12. Guardar pipeline 1──────────────────────────────────────────────────────
joblib.dump(pipeline, MODEL_PATH)
print(f"\n── Pipeline guardado en: {MODEL_PATH}")


── Pipeline guardado en: models\pipeline.pkl


In [78]:
# ── 12. Guardar pipeline 1──────────────────────────────────────────────────────
joblib.dump(pipeline_2, MODEL_PATH)
print(f"\n── Pipeline guardado en: {MODEL_PATH}")


── Pipeline guardado en: models\pipeline.pkl


In [79]:
# ── 13. Verificar pipeline desde .pkl ────────────────────────────────────────
modelo_cargado = joblib.load(MODEL_PATH)
muestra_pred   = modelo_cargado.predict(X_test.iloc[:5])
print(f"  Verificación predict() desde .pkl: {muestra_pred}")

  Verificación predict() desde .pkl: [0 0 0 0 0]


# EXPLICABILIDAD LOGIT - SHAP VALUES

Explica las predicciones del modelo de regresión logística usando valores SHAP (SHapley Additive exPlanations).

In [85]:
# ── 3. Cargar pipeline ────────────────────────────────────────────────────────
pipeline = joblib.load(MODEL_PATH)
print(f"\nPipeline cargado desde: {MODEL_PATH}")

# ── 4. Transformar X con el preprocesador del pipeline ────────────────────────
preprocesador = pipeline.named_steps["preprocesamiento"]
X_transformado = preprocesador.transform(X)

# Recuperar nombres de columnas tras One Hot Encoding
# ohe_features = preprocesador.named_transformers_["cat"]\
#     .get_feature_names_out(categorical_cols).tolist()
feature_names = dummy_cols_2 + numeric_continuous_2

print(f"\nFeatures tras preprocesamiento : {len(feature_names)}")
print(f"  Categóricas (OHE) : {len(dummy_cols_2)}")
print(f"  Numéricas         : {len(numeric_continuous_2)}")


# ── 5. Extraer modelo base del GridSearchCV ───────────────────────────────────
mejor_modelo = pipeline.named_steps["modelo"].best_estimator_


Pipeline cargado desde: models\pipeline.pkl

Features tras preprocesamiento : 7
  Categóricas (OHE) : 2
  Numéricas         : 5


In [86]:
# ── 6. Calcular SHAP values ───────────────────────────────────────────────────
# Muestra de 50 filas para eficiencia
np.random.seed(SEED)
idx_muestra    = np.random.choice(len(X_transformado), size=50, replace=False)
X_muestra      = X_transformado[idx_muestra]

explainer   = shap.LinearExplainer(mejor_modelo, X_muestra)
shap_values = explainer.shap_values(X_muestra)

print(f"\nSHAP values calculados sobre muestra de 50 filas")



SHAP values calculados sobre muestra de 50 filas


In [87]:
print(f"Longitud de nombres de features: {len(feature_names)}")
print(f"Longitud de valores SHAP promedio: {len(np.abs(shap_values).mean(axis=0))}")

Longitud de nombres de features: 7
Longitud de valores SHAP promedio: 7


In [88]:
# ── 7. Importancia global (mean |SHAP|) ───────────────────────────────────────
importancia = pd.DataFrame({
    "feature"   : feature_names,
    "mean_shap" : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_shap", ascending=False).reset_index(drop=True)

print(f"\n── Top 15 variables por importancia SHAP ──────────────────")
print(importancia.head(15).to_string())


── Top 15 variables por importancia SHAP ──────────────────
               feature  mean_shap
0        lifetimevalue   0.370888
1                  Age   0.327758
2    PlanType_Standard   0.168601
3         ltvcac_ratio   0.151898
4       number_helprqs   0.147596
5  number_contentviews   0.066169
6     PlanType_Premium   0.049528


Gráficos SHAP Values

In [89]:
# ── 8. Gráficas ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Análisis SHAP — Churn Prediction", fontsize=13, fontweight="bold")

# 8.1 Bar plot importancia global
top15 = importancia.head(15)
axes[0].barh(top15["feature"][::-1], top15["mean_shap"][::-1], color="#3498db")
axes[0].set_title("Importancia Global (mean |SHAP|)")
axes[0].set_xlabel("mean(|SHAP value|)")

# 8.2 Beeswarm manual (dot plot por feature)
top10_features = importancia.head(10)["feature"].tolist()
top10_idx      = [feature_names.index(f) for f in top10_features]
shap_top10     = shap_values[:, top10_idx]

for i, (feat, shap_col) in enumerate(zip(top10_features, shap_top10.T)):
    y_jitter = np.random.normal(i, 0.08, size=len(shap_col))
    axes[1].scatter(shap_col, y_jitter, alpha=0.3, s=8, c="#e74c3c")

axes[1].set_yticks(range(len(top10_features)))
axes[1].set_yticklabels(top10_features)
axes[1].axvline(0, color="black", lw=0.8, linestyle="--")
axes[1].set_title("Distribución SHAP — Top 10 Features")
axes[1].set_xlabel("SHAP value (impacto en predicción de churn)")

plt.tight_layout()
fig.savefig(FIG_PATH / "03_shap_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nGráfica guardada en: {FIG_PATH / '03_shap_analysis.png'}")


Gráfica guardada en: figures\03_shap_analysis.png


Resultados

In [ ]:
# ── 9. Interpretación resultados ───────────────────────────────────────────────
top3 = importancia.head(3)["feature"].tolist()
print(f"\n── Interpretación de resultados ───────────────────────────")
print(f"  Las 3 variables más influyentes en la predicción de churn:")
for i, feat in enumerate(top3, 1):
    shap_mean = importancia.loc[importancia["feature"]==feat, "mean_shap"].values[0]
    print(f"  {i}. {feat:35s} mean|SHAP|={shap_mean:.4f}")
print(f"""
  Interpretación general:
  - Valores SHAP positivos → aumentan la probabilidad de churn
  - Valores SHAP negativos → disminuyen la probabilidad de churn
  - mean|SHAP| alto → variable muy influyente en la explicabilidad del modelo
""")

# 🅿SEGMENTACIÓN USUARIOS (CLUSTERING)